Data Preprocessing on Breast Cancer Dataset - 5/21/2026 - aprtay2887

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,0.627417
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,0.000000
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,1.000000
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df.isnull().sum()

df = df.fillna(df.median(numeric_only=True))
df = df.drop_duplicates()

concavity_cols = ["mean concavity", "worst concavity"]

scaler = StandardScaler()
df[concavity_cols] = scaler.fit_transform(df[concavity_cols])

X = df.drop("target", axis=1)
Y = df["target"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

In [5]:
from sklearn.svm import SVC

svm_model = SVC()
svm_model.fit(X_train, Y_train)

svm_pred = svm_model.predict(X_test)

In [6]:
from sklearn.metrics import accuracy_score, classification_report

print("SVM Accuracy:", accuracy_score(Y_test, svm_pred))
print("\nClassification Report:\n", classification_report(Y_test, svm_pred))

SVM Accuracy: 0.935672514619883

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.83      0.90        63
           1       0.91      1.00      0.95       108

    accuracy                           0.94       171
   macro avg       0.95      0.91      0.93       171
weighted avg       0.94      0.94      0.93       171



In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"]
}

grid_svm = GridSearchCV(SVC(), param_grid, cv=5)
grid_svm.fit(X_train, Y_train)

print("Best Parameters:", grid_svm.best_params_)

Best Parameters: {'C': 10, 'kernel': 'linear'}


In [9]:
svm_best = grid_svm.best_estimator_
svm_best_pred = svm_best.predict(X_test)

print("Tuned SVM Accuracy:", accuracy_score(Y_test, svm_best_pred))
print("\nClassification Report:\n", classification_report(Y_test, svm_best_pred))

Tuned SVM Accuracy: 0.9707602339181286

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96        63
           1       0.98      0.97      0.98       108

    accuracy                           0.97       171
   macro avg       0.97      0.97      0.97       171
weighted avg       0.97      0.97      0.97       171



In [12]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(eval_metrics="logloss")
xgb_model.fit(X_train, Y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(Y_test, xgb_pred))
print("\nClassification Report:\n", classification_report(Y_test, xgb_pred))

XGBoost Accuracy: 0.9707602339181286

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96        63
           1       0.98      0.97      0.98       108

    accuracy                           0.97       171
   macro avg       0.97      0.97      0.97       171
weighted avg       0.97      0.97      0.97       171



C:\Users\ecpi\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [06:00:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "eval_metrics" } are not used.

  warnings.warn(smsg, UserWarning)


In [14]:
param_grid_xgb = {
    "n_estimators": [50, 100],
    "learning_rate": [0.1, 0.2],
    "max_depth": [3, 4]
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metrics="logloss"),
    param_grid_xgb,
    cv=5
)

grid_xgb.fit(X_train, Y_train)

print("Best XGBoost Parameters:", grid_xgb.best_params_)

xgb_best = grid_xgb.best_estimator_
xgb_best_pred = xgb_best.predict(X_test)

print("Tuned XGBoost Accuracy:", accuracy_score(Y_test, xgb_best_pred))
print("\nClassification Report:\n", classification_report(Y_test, xgb_best_pred))

C:\Users\ecpi\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [06:02:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "eval_metrics" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\ecpi\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [06:02:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "eval_metrics" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\ecpi\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [06:02:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "eval_metrics" } are not used.

  warn

Best XGBoost Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Tuned XGBoost Accuracy: 0.9532163742690059

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.94        63
           1       0.95      0.97      0.96       108

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171



1.) One way could be through sampling. This could over represent certain groups leadining to the model learning patterns that do not generalize. Ultimately, the model would perform well in training but would fail badly in a real-world setting. Anotherw ay could be measurement. This dataset has features that are derived from imaging measurements (like radius, concavity and texture). Measurement bias can happen if the technicians are not following consistent protocols or if different imaging machines give different quality scans. Ultimately, this could lead to differences in feature values that might be unrelated to the disease. Medical bias can be mitigated by improving the overall data collection to ensure there is a diverse representation of patients, collection from multiple hospitals, regions and demographics and standardizing both the equipment and protocols. Another way to mitigate would be to use fairness aware algorithms and still utilize human oversight.

2.) The initial SVM results showed an accuracy of 0.9357 while the tuned SVM showed an accuracy of 0.9708. Between the initial and tuned, the class recalls increased and this shows that the tuning helped SVM performance, especially with class 0 and it went from 0.83 to 0.97. Overall SVM was better at seeing benign tumors and still maintaining great detection. The XGBoost results showed an initial accuracy of 0.9708 and the tuned showed an accuracy of 0.9532. It seems the XGboost model idd worse than the initial version because the accuracy dropped and class 0 recall dropped as well. This means the XGBoost was less effective when tuned at seeing benign tumors. In all, SVM, tuned version, did better performance wise as it got the highest accuracy and showed a more balanced performance overall.